# Overnight Experiment Sweep

Runs a grid of experiments back-to-back, saving per-config results to
`../results/sweep_overnight.json` after each config — so partial results survive
a kernel crash. Re-running the notebook skips configs already in the JSON.

**Configs (all paper-convention metric, leak-free Subset pipeline, 10-fold GroupKFold,
30 epochs, batch 16, LR 1e-4):**

| # | id | imbalance | softpow | model |
|---|---|---|---|---|
| 1 | `loss-p05` | weighted loss | 0.5 | baseline ViT |
| 2 | `loss-p025` | weighted loss | 0.25 | baseline ViT |
| 3 | `sampler-p05` | weighted sampler only | 0.5 | baseline ViT |
| 4 | `kan-last-p05` | weighted loss | 0.5 | KAN @ last block |
| 5 | `kan-first-p05` | weighted loss | 0.5 | KAN @ first block |
| 6 | `kan-last2-p05` | weighted loss | 0.5 | KAN @ last 2 blocks |

Per-config: per-fold log line + JSON dump (per-fold metrics, per-fold mean ± std,
pooled aggregate, elapsed time). Final summary table at the end.


In [1]:
# --- Imports + data setup (runs once) ---
import os, json, time, copy, gc

import numpy as np
import pandas as pd
import torch
import torch.optim as optim
import torch.nn as nn
import matplotlib.pyplot as plt
from torch.utils.data import Dataset, DataLoader, Subset, WeightedRandomSampler
from sklearn.model_selection import GroupKFold
from sklearn.utils.class_weight import compute_class_weight

from cochleogram_vit.models.vit import CochleogramViT
from cochleogram_vit.models.vit_kan import CochleogramViTKAN


DATA_DIR      = '../data/processed/cochleograms'
METADATA_PATH = '../data/processed/metadata.csv'
RESULTS_PATH  = '../results/sweep_overnight.json'
BATCH_SIZE    = 16
EPOCHS        = 30
LEARNING_RATE = 1e-4


class CochleogramDataset(Dataset):
    def __init__(self, data_dir, metadata_path):
        self.data_dir = data_dir
        self.metadata = pd.read_csv(metadata_path)
        self._viridis = plt.get_cmap('viridis')

    def __len__(self):
        return len(self.metadata)

    def __getitem__(self, idx):
        row = self.metadata.iloc[idx]
        npy_path = os.path.join(self.data_dir, os.path.basename(row['npy_path']))
        coch = np.load(npy_path)  # [0,1]
        rgb = self._viridis(coch)[:, :, :3].transpose(2, 0, 1)
        return torch.from_numpy(np.ascontiguousarray(rgb)).float(), int(row['label'])


dataset = CochleogramDataset(DATA_DIR, METADATA_PATH)
metadata = dataset.metadata.copy()
metadata['patient_id'] = metadata['npy_path'].apply(lambda x: os.path.basename(x).split('_')[0])

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Dataset size: {len(dataset)}  device: {device}')

gkf = GroupKFold(n_splits=10)
FOLDS = list(gkf.split(metadata, groups=metadata['patient_id'].values))
print(f'GroupKFold: {len(FOLDS)} folds')

os.makedirs(os.path.dirname(RESULTS_PATH), exist_ok=True)


Dataset size: 6898  device: cuda
GroupKFold: 10 folds


In [2]:
# --- Experiment grid (edit here to add/remove configs) ---
EXPERIMENTS = [
    # softening lever on single (weighted-loss-only) baseline
    {'id': 'loss-p05',      'imbalance': 'weighted_loss',    'softpow': 0.5,  'kan_blocks': None},
    {'id': 'loss-p025',     'imbalance': 'weighted_loss',    'softpow': 0.25, 'kan_blocks': None},
    # sampler-only variant (correctly scoped) at the sweet-spot softening
    {'id': 'sampler-p05',   'imbalance': 'weighted_sampler', 'softpow': 0.5,  'kan_blocks': None},
    # KAN placement sweep at the sweet-spot setup (loss-p05)
    {'id': 'kan-last-p05',  'imbalance': 'weighted_loss',    'softpow': 0.5,  'kan_blocks': (-1,)},
    {'id': 'kan-first-p05', 'imbalance': 'weighted_loss',    'softpow': 0.5,  'kan_blocks': (0,)},
    {'id': 'kan-last2-p05', 'imbalance': 'weighted_loss',    'softpow': 0.5,  'kan_blocks': (-2, -1)},
]
print(f'Sweep size: {len(EXPERIMENTS)} configs')
for e in EXPERIMENTS:
    print(f"  - {e['id']:<16}  imbalance={e['imbalance']:<18} softpow={e['softpow']:<4}  kan_blocks={e['kan_blocks']}")


Sweep size: 6 configs
  - loss-p05          imbalance=weighted_loss      softpow=0.5   kan_blocks=None
  - loss-p025         imbalance=weighted_loss      softpow=0.25  kan_blocks=None
  - sampler-p05       imbalance=weighted_sampler   softpow=0.5   kan_blocks=None
  - kan-last-p05      imbalance=weighted_loss      softpow=0.5   kan_blocks=(-1,)
  - kan-first-p05     imbalance=weighted_loss      softpow=0.5   kan_blocks=(0,)
  - kan-last2-p05     imbalance=weighted_loss      softpow=0.5   kan_blocks=(-2, -1)


In [3]:
# --- Helpers ---
def softened_weights(softpow):
    raw = compute_class_weight('balanced', classes=np.array([0,1,2,3]), y=metadata['label'].values)
    w = raw ** softpow
    return w / w.sum() * len(w)


def build_model(kan_blocks):
    if kan_blocks is None:
        return CochleogramViT(
            image_size=128, patch_size=16, num_classes=4,
            dim=512, depth=6, heads=8, mlp_dim=1024, channels=3,
            dropout=0.3, emb_dropout=0.2,
        ).to(device)
    return CochleogramViTKAN(
        image_size=128, patch_size=16, num_classes=4,
        dim=512, depth=6, heads=8, mlp_dim=1024, channels=3,
        dropout=0.3, emb_dropout=0.2,
        kan_blocks=kan_blocks, grid_size=5, spline_order=3,
    ).to(device)


def build_optimizer(model, kan_blocks):
    if kan_blocks is None:
        return optim.Adam(model.parameters(), lr=LEARNING_RATE, weight_decay=1e-4)
    # KAN spline params on a separate, lower-LR group (0.1x)
    kan_ids = {id(p) for p in model.kan_parameters()}
    base = [p for p in model.parameters() if id(p) not in kan_ids]
    kan  = list(model.kan_parameters())
    return optim.Adam(
        [{'params': base, 'lr': LEARNING_RATE},
         {'params': kan,  'lr': LEARNING_RATE * 0.1}],
        weight_decay=1e-4,
    )


def lr_lambda(epoch):
    warmup = 4
    if epoch < warmup:
        return (epoch + 1) / warmup
    denom = EPOCHS - warmup
    return 0.5 * (1 + np.cos(np.pi * (epoch - warmup) / denom)) if denom > 0 else 0.0


def paper_metrics(preds, labels):
    p = np.asarray(preds); y = np.asarray(labels)
    TP   = int(np.sum((y != 0) & (p == y)))
    FN   = int(np.sum((y != 0) & (p == 0)))
    FN_w = int(np.sum((y != 0) & (p != 0) & (p != y)))
    TN   = int(np.sum((y == 0) & (p == 0)))
    FP   = int(np.sum((y == 0) & (p != 0)))
    TP_b = TP + FN_w  # paper: any adventitious flagged adventitious
    se = TP_b / (TP_b + FN + 1e-8)
    sp = TN / (TN + FP + 1e-8)
    return {'TP': TP, 'FN': FN, 'FN_wrong': FN_w, 'TN': TN, 'FP': FP,
            'se': float(se), 'sp': float(sp), 'score': float((se + sp) / 2)}


def evaluate(model, loader):
    model.eval()
    preds, labels = [], []
    with torch.no_grad():
        for x, y in loader:
            x = x.to(device)
            preds.extend(model(x).argmax(dim=1).cpu().numpy().tolist())
            labels.extend(y.numpy().tolist())
    return preds, labels


def run_config(cfg):
    print(f"\n{'=' * 70}\n[{cfg['id']}]  imbalance={cfg['imbalance']}  softpow={cfg['softpow']}  kan_blocks={cfg['kan_blocks']}\n{'=' * 70}")
    t0 = time.time()

    cw = softened_weights(cfg['softpow'])
    cw_t = torch.tensor(cw, dtype=torch.float).to(device)
    print(f"  class weights (^{cfg['softpow']}): {np.round(cw, 3).tolist()}")

    fold_rows, pooled_preds, pooled_labels = [], [], []

    for fold, (train_idx, val_idx) in enumerate(FOLDS):
        torch.manual_seed(42 + fold); np.random.seed(42 + fold)
        if torch.cuda.is_available():
            torch.cuda.manual_seed_all(42 + fold)

        train_subset = Subset(dataset, train_idx)
        val_subset   = Subset(dataset, val_idx)
        train_labels = metadata['label'].values[train_idx]

        if cfg['imbalance'] == 'weighted_sampler':
            sw = torch.tensor([cw[l] for l in train_labels], dtype=torch.float)
            sampler = WeightedRandomSampler(weights=sw, num_samples=len(sw), replacement=True)
            train_loader = DataLoader(train_subset, batch_size=BATCH_SIZE, sampler=sampler)
            criterion = nn.CrossEntropyLoss()
        else:  # weighted_loss only
            train_loader = DataLoader(train_subset, batch_size=BATCH_SIZE, shuffle=True)
            criterion = nn.CrossEntropyLoss(weight=cw_t)
        val_loader = DataLoader(val_subset, batch_size=BATCH_SIZE, shuffle=False)

        model = build_model(cfg['kan_blocks'])
        opt   = build_optimizer(model, cfg['kan_blocks'])
        sched = optim.lr_scheduler.LambdaLR(opt, lr_lambda)

        best_score = -1.0
        best_state = None
        best_epoch = 0

        for epoch in range(EPOCHS):
            model.train()
            for x, y in train_loader:
                x, y = x.to(device), y.to(device)
                opt.zero_grad()
                loss = criterion(model(x), y)
                loss.backward()
                if cfg['kan_blocks'] is not None:
                    torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                opt.step()

            preds, labels = evaluate(model, val_loader)
            m = paper_metrics(preds, labels)
            if m['score'] > best_score:
                best_score = m['score']
                best_state = copy.deepcopy(model.state_dict())
                best_epoch = epoch + 1
            sched.step()

        # final eval at best checkpoint
        model.load_state_dict(best_state)
        preds, labels = evaluate(model, val_loader)
        m = paper_metrics(preds, labels)
        fold_rows.append({'fold': fold + 1, 'best_epoch': best_epoch, **m})
        pooled_preds.extend(preds); pooled_labels.extend(labels)
        print(f"  fold {fold+1:>2}: best_ep={best_epoch:>2}  Se={m['se']*100:5.2f}  Sp={m['sp']*100:5.2f}  Score={m['score']*100:5.2f}")

        del model, opt, sched, train_loader, val_loader
        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()

    pooled = paper_metrics(pooled_preds, pooled_labels)
    se_mean = float(np.mean([r['se']    for r in fold_rows]))
    sp_mean = float(np.mean([r['sp']    for r in fold_rows]))
    sc_mean = float(np.mean([r['score'] for r in fold_rows]))
    sc_std  = float(np.std ([r['score'] for r in fold_rows]))
    elapsed = time.time() - t0

    print(f"  PER-FOLD MEAN: Se={se_mean*100:.2f}  Sp={sp_mean*100:.2f}  Score={sc_mean*100:.2f} (std {sc_std*100:.2f})")
    print(f"  POOLED       : Se={pooled['se']*100:.2f}  Sp={pooled['sp']*100:.2f}  Score={pooled['score']*100:.2f}")
    print(f"  elapsed: {elapsed/60:.1f} min")

    return {
        'id': cfg['id'],
        'config': {**cfg, 'kan_blocks': list(cfg['kan_blocks']) if cfg['kan_blocks'] else None},
        'elapsed_sec': elapsed,
        'folds': fold_rows,
        'per_fold_mean': {'se': se_mean, 'sp': sp_mean, 'score': sc_mean, 'score_std': sc_std},
        'pooled_aggregate': pooled,
    }


In [4]:
# --- Sweep loop (resume-safe) ---
results = []
if os.path.exists(RESULTS_PATH):
    try:
        results = json.load(open(RESULTS_PATH))
        print(f'Loaded {len(results)} prior results from {RESULTS_PATH}')
    except Exception as e:
        print(f'Could not load prior results ({e}); starting fresh.')
        results = []
done_ids = {r.get('id') for r in results}
print(f'Already done: {sorted(done_ids)}')

for cfg in EXPERIMENTS:
    if cfg['id'] in done_ids:
        print(f"\n[{cfg['id']}] already done, skipping.")
        continue
    try:
        r = run_config(cfg)
        results.append(r)
    except KeyboardInterrupt:
        print(f"\n[{cfg['id']}] interrupted by user; saving and stopping.")
        with open(RESULTS_PATH, 'w') as f:
            json.dump(results, f, indent=2, default=str)
        raise
    except Exception as e:
        print(f"\n[{cfg['id']}] FAILED: {type(e).__name__}: {e}")
        results.append({'id': cfg['id'], 'config': cfg, 'error': f'{type(e).__name__}: {e}'})
    with open(RESULTS_PATH, 'w') as f:
        json.dump(results, f, indent=2, default=str)
    print(f"  -> wrote {RESULTS_PATH}")

print('\nSWEEP COMPLETE')


Loaded 4 prior results from ../results/sweep_overnight.json
Already done: ['kan-last-p05', 'loss-p025', 'loss-p05', 'sampler-p05']

[loss-p05] already done, skipping.

[loss-p025] already done, skipping.

[sampler-p05] already done, skipping.

[kan-last-p05] already done, skipping.

[kan-first-p05]  imbalance=weighted_loss  softpow=0.5  kan_blocks=(0,)
  class weights (^0.5): [0.563, 0.787, 1.141, 1.51]
[CochleogramViTKAN] Parameters — total: 17,758,724  trainable: 17,758,724  KAN: 5,242,880  (kan_blocks=(0,))
  fold  1: best_ep= 4  Se=73.96  Sp=68.72  Score=71.34
[CochleogramViTKAN] Parameters — total: 17,758,724  trainable: 17,758,724  KAN: 5,242,880  (kan_blocks=(0,))
  fold  2: best_ep= 6  Se=70.52  Sp=69.50  Score=70.01
[CochleogramViTKAN] Parameters — total: 17,758,724  trainable: 17,758,724  KAN: 5,242,880  (kan_blocks=(0,))
  fold  3: best_ep= 8  Se=62.83  Sp=81.02  Score=71.93
[CochleogramViTKAN] Parameters — total: 17,758,724  trainable: 17,758,724  KAN: 5,242,880  (kan_block

KeyboardInterrupt: 

In [ ]:
# --- Summary table ---
results = json.load(open(RESULTS_PATH))
hdr = f"{'id':<18} {'per-fold (Se/Sp/Score±std)':<34}  {'pooled (Se/Sp/Score)':<28}  {'time':<6}"
print(hdr); print('-' * len(hdr))
for r in results:
    if 'error' in r:
        print(f"{r['id']:<18} ERROR: {r['error']}")
        continue
    m, p = r['per_fold_mean'], r['pooled_aggregate']
    print(f"{r['id']:<18} {m['se']*100:5.1f}/{m['sp']*100:5.1f}/{m['score']*100:5.1f}±{m['score_std']*100:4.1f}            "
          f"{p['se']*100:5.1f}/{p['sp']*100:5.1f}/{p['score']*100:5.1f}            "
          f"{r['elapsed_sec']/60:5.1f}m")
